In [ ]:
# 02: Test-set prep and NDWI baseline (Project 11)
#**Name:** Devon Clermont-Gauer

#1. Computes per-band statistics from the training tiles (used to rescale bands for training).
#2. Prepares test tiles for correction in labelme: RGB images + draft polygons from the Dynamic World masks.
#3. Converts corrected labelme files back into masks.
#4. Scores a simple NDWI threshold on the test set (IoU, F1): the baseline the U-Net must beat.

#**How to run:** Cells 1–3 and 5 any time. Cell 4 after correcting tiles in labelme, then re-run Cell 5.

In [ ]:
# Cell 1: connect to Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json, random
from collections import defaultdict
import numpy as np
import rasterio
import cv2                       # OpenCV: turning masks into polygons and back
from PIL import Image

OUT   = '/content/drive/MyDrive/water_dataset'      # tiles from notebook 01
LM    = '/content/drive/MyDrive/water_labelme'      # files for labelme
CORR  = f'{OUT}/test/masks_corrected'               # corrected masks (after labelme)
BANDS = ['B2', 'B3', 'B4', 'B8', 'B11']             # blue, green, red, NIR, SWIR

def region_of(path):
    """Tile file name -> region name, e.g. 'kejimkujik_r256_c512.tif' -> 'kejimkujik'."""
    return os.path.basename(path).rsplit('_r', 1)[0]

print('Train tiles:', len(glob.glob(f'{OUT}/train/images/*.tif')))
print('Test tiles: ', len(glob.glob(f'{OUT}/test/images/*.tif')))

In [ ]:
# Cell 2: mean and standard deviation of each band, from TRAINING tiles only
# (using val/test tiles here would leak information about them into training)
sums, sq, n = np.zeros(5), np.zeros(5), 0
for f in glob.glob(f'{OUT}/train/images/*.tif'):
    with rasterio.open(f) as s:
        a = s.read().reshape(5, -1).astype(np.float64)
    a = a[:, (a != 0).any(axis=0)]          # ignore empty (no-data) pixels
    sums += a.sum(axis=1)
    sq   += (a ** 2).sum(axis=1)
    n    += a.shape[1]

mean = sums / n
std  = np.sqrt(sq / n - mean ** 2)
stats = {'bands': BANDS, 'mean': mean.round(2).tolist(), 'std': std.round(2).tolist()}
json.dump(stats, open(f'{OUT}/band_stats.json', 'w'), indent=2)

for b, m, s in zip(BANDS, mean, std):
    print(f'{b:4}  mean {m:8.1f}   std {s:8.1f}')
print('Saved:', f'{OUT}/band_stats.json')

In [ ]:
# Cell 3: make RGB images + draft polygons (from the existing masks) for labelme
# Outer water edges -> 'water' polygons; islands inside water -> 'land' polygons.
random.seed(0)                           # same sample every run
PER_REGION = 25                          # up to 25 tiles per test region
os.makedirs(LM, exist_ok=True)

by_region = defaultdict(list)
for f in sorted(glob.glob(f'{OUT}/test/images/*.tif')):
    by_region[region_of(f)].append(f)

chosen = []
for region, fs in by_region.items():
    chosen += random.sample(fs, min(PER_REGION, len(fs)))

for f in chosen:
    name = os.path.basename(f)[:-4]

    # RGB image for labelme (natural colour, brightened, 8-bit PNG)
    with rasterio.open(f) as s:
        rgb = s.read([3, 2, 1]).transpose(1, 2, 0)
    Image.fromarray((np.clip(rgb / 3000, 0, 1) * 255).astype(np.uint8)).save(f'{LM}/{name}.png')

    # Draft polygons traced from the Dynamic World mask
    with rasterio.open(f.replace('/images/', '/masks/')) as s:
        mask = s.read(1).astype(np.uint8)
    contours, hier = cv2.findContours(mask, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    shapes = []
    for i, c in enumerate(contours):
        c = cv2.approxPolyDP(c, 1.0, True)          # simplify the outline slightly
        if len(c) < 3:                              # skip specks too small to be a polygon
            continue
        label = 'water' if hier[0][i][3] == -1 else 'land'   # no parent = water edge; hole = island
        shapes.append({'label': label, 'points': c.reshape(-1, 2).astype(float).tolist(),
                       'group_id': None, 'shape_type': 'polygon', 'flags': {}})

    json.dump({'version': '5.4.1', 'flags': {}, 'shapes': shapes,
               'imagePath': f'{name}.png', 'imageData': None,
               'imageHeight': mask.shape[0], 'imageWidth': mask.shape[1]},
              open(f'{LM}/{name}.json', 'w'))

print(f'Prepared {len(chosen)} tiles for labelme in {LM}')
for region, fs in by_region.items():
    print(f'  {region}: {min(PER_REGION, len(fs))} of {len(fs)} tiles')

In [ ]:
# Cell 4: turn corrected labelme polygons back into mask GeoTIFFs
# Water polygons are filled with 1, then land (island) polygons are cut back to 0.
os.makedirs(CORR, exist_ok=True)
done = 0
for jf in sorted(glob.glob(f'{LM}/*.json')):
    d = json.load(open(jf))
    name = os.path.basename(jf)[:-5]
    m = np.zeros((d['imageHeight'], d['imageWidth']), np.uint8)
    for label, value in [('water', 1), ('land', 0)]:
        for sh in d['shapes']:
            if sh['label'] == label:
                pts = np.round(np.array(sh['points'])).astype(np.int32)
                cv2.fillPoly(m, [pts], value)

    # Copy the original mask's location info so the corrected mask stays georeferenced
    with rasterio.open(f'{OUT}/test/masks/{name}.tif') as s:
        prof = s.profile
    with rasterio.open(f'{CORR}/{name}.tif', 'w', **prof) as dst:
        dst.write(m[None].astype(prof['dtype']))
    done += 1

print(f'Saved {done} corrected masks to {CORR}')

In [ ]:
# Cell 5: score a simple NDWI threshold on the test tiles
# NDWI = (Green - NIR) / (Green + NIR); water if NDWI > 0  (McFeeters, 1996)
THRESH = 0.0

def ndwi_predict(img):
    g, nir = img[1].astype(float), img[3].astype(float)      # B3 = green, B8 = NIR
    return (g - nir) / (g + nir + 1e-6) > THRESH

def evaluate(mask_dir, label):
    """Pool true/false positives over all tiles that have a mask in mask_dir."""
    totals = defaultdict(lambda: np.zeros(3))                # per region: TP, FP, FN
    for mf in sorted(glob.glob(f'{mask_dir}/*.tif')):
        name = os.path.basename(mf)
        with rasterio.open(f'{OUT}/test/images/{name}') as s:
            img = s.read()
        with rasterio.open(mf) as s:
            true = s.read(1) == 1
        valid = (img != 0).any(axis=0)                       # ignore empty pixels
        pred = ndwi_predict(img) & valid
        true = true & valid
        tp, fp, fn = (pred & true).sum(), (pred & ~true).sum(), (~pred & true).sum()
        totals[region_of(mf)] += [tp, fp, fn]
        totals['ALL'] += [tp, fp, fn]

    print(f'\nNDWI vs {label}')
    print(f"{'region':12} {'IoU':>6} {'F1':>6}")
    results = {}
    for r, (tp, fp, fn) in sorted(totals.items()):
        iou = tp / (tp + fp + fn) if tp + fp + fn else 0
        f1  = 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0
        results[r] = {'iou': round(iou, 3), 'f1': round(f1, 3)}
        print(f'{r:12} {iou:6.3f} {f1:6.3f}')
    return results

report = {'threshold': THRESH,
          'vs_dynamic_world': evaluate(f'{OUT}/test/masks', 'Dynamic World masks (all test tiles)')}
if glob.glob(f'{CORR}/*.tif'):
    report['vs_corrected'] = evaluate(CORR, 'corrected masks (labelme)')
else:
    print('\nNo corrected masks yet: run Cell 4 after labelme, then re-run this cell.')

os.makedirs(f'{OUT}/results', exist_ok=True)
json.dump(report, open(f'{OUT}/results/ndwi_baseline.json', 'w'), indent=2)
print('\nSaved:', f'{OUT}/results/ndwi_baseline.json')